# SAM2 Video Segmentation
**Automatic Surgical Tool Segmentation in Video**

## 1. Installation
Run **only once** to install SAM2 and ffmpeg.

In [ ]:
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .

# Install ffmpeg (Ubuntu):
# !sudo apt-get install ffmpeg -y

In [ ]:
!pip install opencv-python

## 2. Download Checkpoint

In [ ]:
import urllib.request, os

ckpt_dir  = r"\\wsl.localhost\Ubuntu\home\mci\BHS_project\checkpoints"  # Change this to your desired checkpoint directory
ckpt_path = os.path.join(ckpt_dir, "sam2_hiera_large.pt")
ckpt_url  = "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt"

if not os.path.exists(ckpt_path):
    print("Descargando checkpoint (~900 MB)...")
    urllib.request.urlretrieve(ckpt_url, ckpt_path)
    print("Descarga completada:", ckpt_path)
else:
    print("Checkpoint ya existe:", ckpt_path)

## 3. Imports

In [ ]:
import subprocess
import sys

# Ensure SAM2 is installed in the current kernel
try:
    import sam2
    print("✓ SAM2 already installed")
except ImportError:
    print("Installing SAM2...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/sam2.git"])
    print("✓ SAM2 installed successfully")

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from sam2.build_sam import build_sam2_video_predictor

## 4. Configuration

In [ ]:
VIDEO_PATH = r"\\wsl.localhost\Ubuntu\home\mci\BHS_project\videoSegmentation\2026-05-20_10-24-38 - Copy.mp4"  # change that to your video path

FRAMES_DIR = "frames"
MASKS_DIR  = "masks"

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(MASKS_DIR,  exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## 5. Extract Video Frames

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_path = os.path.join(FRAMES_DIR, f"{frame_idx:05d}.jpg")
    cv2.imwrite(frame_path, frame)
    frame_idx += 1

cap.release()
print("Extracted frames:", frame_idx)

## 6. Load SAM2

In [ ]:
checkpoint = r"\\wsl.localhost\Ubuntu\home\mci\BHS_project\checkpoints\sam2_hiera_large.pt"  # Change this to your checkpoint path
model_cfg  = "configs/sam2/sam2_hiera_l.yaml"

predictor = build_sam2_video_predictor(
    model_cfg,
    checkpoint,
    device=DEVICE
)
print("SAM2 loaded")

## 7. Initialize Video State

In [ ]:
inference_state = predictor.init_state(video_path=FRAMES_DIR)
print("Inference state initialized")

## 8. Display First Frame

In [ ]:
frame_names = sorted(os.listdir(FRAMES_DIR))

first_frame = np.array(
    Image.open(os.path.join(FRAMES_DIR, frame_names[0]))
)

plt.figure(figsize=(10, 10))
plt.imshow(first_frame)
plt.title("First frame")
plt.axis("off")
plt.show()

## 9. Define Point on Surgical Tool



In [ ]:
# Show image with DENSE COORDINATE GRID with VISIBLE NUMBERS
fig, ax = plt.subplots(figsize=(18, 12))
ax.imshow(first_frame, alpha=0.7)  # Slightly transparent to see grid better
ax.set_title("COORDINATE GRID - Hover to see X,Y | Take note of coordinates for each tool", 
             fontsize=16, fontweight='bold', pad=20, color='white',
             bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

# Get image dimensions
img_height, img_width = first_frame.shape[:2]

# Add DENSE grid with major and minor ticks
ax.set_xlim(0, img_width)
ax.set_ylim(img_height, 0)  # Flip Y axis to match image coordinates

# Major grid every 100 pixels
major_ticks_x = np.arange(0, img_width, 100)
major_ticks_y = np.arange(0, img_height, 100)
ax.set_xticks(major_ticks_x, minor=False)
ax.set_yticks(major_ticks_y, minor=False)

# Minor grid every 50 pixels
minor_ticks_x = np.arange(0, img_width, 50)
minor_ticks_y = np.arange(0, img_height, 50)
ax.set_xticks(minor_ticks_x, minor=True)
ax.set_yticks(minor_ticks_y, minor=True)

# Style grids with HIGH VISIBILITY
ax.grid(which='major', color='white', linestyle='-', linewidth=2.5, alpha=0.8)
ax.grid(which='minor', color='lime', linestyle='--', linewidth=1.2, alpha=0.6)

# LARGE, BOLD, WHITE numbers with BLACK outline for visibility
for label in ax.get_xticklabels(which='major'):
    label.set_fontsize(14)
    label.set_fontweight('bold')
    label.set_color('white')
    label.set_bbox(dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.8, edgecolor='white', linewidth=1))

for label in ax.get_yticklabels(which='major'):
    label.set_fontsize(14)
    label.set_fontweight('bold')
    label.set_color('white')
    label.set_bbox(dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.8, edgecolor='white', linewidth=1))

# Minor tick labels (smaller but visible)
for label in ax.get_xticklabels(which='minor'):
    label.set_fontsize(10)
    label.set_color('lime')
    label.set_bbox(dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

for label in ax.get_yticklabels(which='minor'):
    label.set_fontsize(10)
    label.set_color('lime')
    label.set_bbox(dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

ax.tick_params(axis='both', which='major', length=12, width=2.5, colors='white')
ax.tick_params(axis='both', which='minor', length=6, width=1.5, colors='lime')

ax.set_xlabel(f'X Coordinate (pixels) - 0 to {img_width}', fontsize=14, fontweight='bold', color='white',
              bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
ax.set_ylabel(f'Y Coordinate (pixels) - 0 to {img_height}', fontsize=14, fontweight='bold', color='white',
              bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))

# Add info boxes
ax.text(10, 30, f'Width: {img_width}px', fontsize=13, color='white', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='black', alpha=0.85, edgecolor='yellow', linewidth=2))
ax.text(img_width-250, 30, f'Height: {img_height}px', fontsize=13, color='white', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='black', alpha=0.85, edgecolor='yellow', linewidth=2))

# Add legend
ax.text(img_width-450, img_height-50, 'White grid=100px | Lime grid=50px', fontsize=12, color='white', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='black', alpha=0.85, edgecolor='cyan', linewidth=2))

# Enable coordinates on hover
def on_move(event):
    if event.xdata is not None and event.ydata is not None:
        x, y = int(event.xdata), int(event.ydata)
        ax.set_title(f"COORDINATE GRID | Current Position: X={x}, Y={y} | White grid=100px, Lime grid=50px", 
                     fontsize=16, fontweight='bold', pad=20, color='white',
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
        fig.canvas.draw_idle()

fig.canvas.mpl_connect('motion_notify_event', on_move)
plt.tight_layout()
plt.show()

print("✓ HIGH-VISIBILITY COORDINATE GRID")
print(f"  Image dimensions: {img_width} x {img_height} pixels")
print(f"  WHITE grid lines every 100 pixels (major)")
print(f"  LIME grid lines every 50 pixels (minor)")
print(f"  Numbers are large, bold, and have black backgrounds for visibility")
print("  Hover over image to see exact X,Y coordinates")

In [ ]:
# *** DEFINE MULTIPLE TOOLS ***
# Format: [tool_name] = (X, Y)
tools = {
    "Tool 1": (1000, 600),
    "Tool 2": (500, 400),
    # Add more tools as needed:
    # "Tool 3": (x_coord, y_coord),
}

clicked_points = [[x, y] for x, y in tools.values()]

# Visualization with all selected points
plt.figure(figsize=(14, 9))
plt.imshow(first_frame)
colors = plt.cm.tab20(np.linspace(0, 1, len(tools)))
for idx, (name, (x, y)) in enumerate(tools.items()):
    plt.scatter([x], [y], c=[colors[idx]], s=300, zorder=5, label=name)
plt.title(f"Selected {len(tools)} tools")
plt.legend()
plt.axis("off")
plt.show()

print(f"Tools selected: {', '.join(tools.keys())}")

In [ ]:
%matplotlib notebook

# INTERACTIVE: Click on each tool to select it
# Click multiple times for multiple tools, then close the plot to finish
# If you don't click, it will keep the previously defined tools

# Check if clicked_points exists from previous cell
if 'clicked_points' not in locals():
    print("⚠️  WARNING: clicked_points not initialized!")
    print("   Please run the previous cell (Define Multiple Tools) FIRST")
    print("   Then run this cell again")
    clicked_points = []
else:
    print(f"✓ Previous tools loaded: {len(clicked_points)} tools")

clicked_points_interactive = []

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(first_frame)
ax.set_title("CLICK on each tool to select it (close plot when done)", fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_xlabel('X Coordinate (pixels)', fontsize=11)
ax.set_ylabel('Y Coordinate (pixels)', fontsize=11)

scatter = ax.scatter([], [], c='red', s=200, zorder=5, edgecolors='yellow', linewidth=2)
text_labels = []

def on_click(event):
    if event.inaxes != ax:
        return
    x, y = int(event.xdata), int(event.ydata)
    clicked_points_interactive.append([x, y])
    
    # Update scatter plot
    scatter.set_offsets(clicked_points_interactive)
    
    # Add text label with tool number
    tool_num = len(clicked_points_interactive)
    text = ax.text(x+20, y+20, f'Tool {tool_num}\n({x},{y})', 
                   color='yellow', fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='red', alpha=0.7))
    text_labels.append(text)
    
    ax.set_title(f"CLICKED {tool_num} tool(s) - Click more or close plot to finish", 
                 fontsize=12, fontweight='bold')
    fig.canvas.draw()
    print(f"✓ Tool {tool_num} saved at ({x}, {y})")

fig.canvas.mpl_connect('button_press_event', on_click)
plt.tight_layout()
plt.show()

# IMPORTANT: Only update clicked_points if you actually clicked on tools
if len(clicked_points_interactive) > 0:
    clicked_points = clicked_points_interactive
    print(f"\n✓ INTERACTIVE selection updated!")
    print(f"  Total tools selected: {len(clicked_points)}")
    for i, (x, y) in enumerate(clicked_points, 1):
        print(f"  Tool {i}: ({x}, {y})")
else:
    if len(clicked_points) == 0:
        print(f"\n⚠️  No tools defined!")
        print(f"   Please run the previous cell (Define Multiple Tools) to define tools first")
    else:
        print(f"\n✓ No interactive clicks detected - keeping previous tool definition")
        print(f"  Current tools: {len(clicked_points)}")
        for i, (x, y) in enumerate(clicked_points, 1):
            print(f"  Tool {i}: ({x}, {y})")

## 10. Add Prompt

In [ ]:
ann_frame_idx = 0

if len(clicked_points) == 0:
    raise RuntimeError("No tools defined. Define tools in previous cell first.")

# Process each tool separately
for tool_idx, (x, y) in enumerate(clicked_points, start=1):
    ann_obj_id = tool_idx  # Each tool gets a unique ID
    
    points = np.array([[x, y]], dtype=np.float32)
    labels = np.array([1], np.int32)
    
    _, out_obj_ids, out_mask_logits = predictor.add_new_points(
        inference_state=inference_state,
        frame_idx=ann_frame_idx,
        obj_id=ann_obj_id,
        points=points,
        labels=labels,
    )
    print(f"Tool {tool_idx}: Prompt added at ({x}, {y})")

print(f"Total tools added: {len(clicked_points)}")

## 11. Display Initial Mask

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(first_frame)

# Display masks for all tools with different colors
colors = plt.cm.tab20(np.linspace(0, 1, len(clicked_points)))
for tool_idx in range(1, len(clicked_points) + 1):
    if tool_idx in [out_obj_ids[i] for i in range(len(out_obj_ids))]:
        # Find the mask index
        mask_idx = list(out_obj_ids).index(tool_idx) if tool_idx in out_obj_ids else None
        if mask_idx is not None:
            mask = (out_mask_logits[mask_idx] > 0.0).cpu().numpy()
            if mask.ndim == 3 and mask.shape[0] == 1:
                mask = mask[0]
            plt.imshow(mask, alpha=0.4, cmap='Greys')

plt.title(f"Initial masks for {len(clicked_points)} tools")
plt.axis("off")
plt.show()

## 12. Propagate Through Video, Save Outputs & Export Results

In [ ]:
print("=" * 60)
print("STEP 1: Propagating segmentation through video...")
print("=" * 60)

video_segments = {}
frame_count = 0

for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }
    frame_count += 1

print(f"✓ Segmentation completed for {frame_count} frames\n")

# ============================================================================
print("=" * 60)
print("STEP 2: Saving raw frames and instance masks...")
print("=" * 60)

NOTEBOOK_DIR = r"/home/mci/BHS_project/sam2-surgical-segmentation"
OUTPUT_FRAMES_DIR = os.path.join(NOTEBOOK_DIR, "output", "frames")
OUTPUT_MASKS_DIR = os.path.join(NOTEBOOK_DIR, "output", "masks")

os.makedirs(OUTPUT_FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASKS_DIR, exist_ok=True)

frames_saved = 0
masks_saved = 0

for frame_idx, frame_name in enumerate(frame_names):
    frame_img = Image.open(os.path.join(FRAMES_DIR, frame_name)).convert("RGB")
    frame_img.save(os.path.join(OUTPUT_FRAMES_DIR, f"frame_{frame_idx:05d}.jpg"))
    frames_saved += 1

    w, h = frame_img.size
    instance_mask = np.zeros((h, w), dtype=np.uint8)

    for obj_id, mask in video_segments.get(frame_idx, {}).items():
        mask = np.asarray(mask)
        if mask.ndim == 3 and mask.shape[0] == 1:
            mask = mask[0]
        instance_mask[mask] = obj_id

    Image.fromarray(instance_mask, mode="L").save(
        os.path.join(OUTPUT_MASKS_DIR, f"frame_{frame_idx:05d}.png")
    )
    masks_saved += 1

print(f"✓ Saved {frames_saved} frames to '{OUTPUT_FRAMES_DIR}'")
print(f"✓ Saved {masks_saved} instance masks to '{OUTPUT_MASKS_DIR}'\n")

# ============================================================================
print("=" * 60)
print("STEP 3: Saving individual tool masks...")
print("=" * 60)

for frame_idx, segments in video_segments.items():
    for obj_id, mask in segments.items():
        mask_uint8 = (mask * 255).astype(np.uint8)
        save_path = os.path.join(MASKS_DIR, f"{frame_idx:05d}.png")
        cv2.imwrite(save_path, mask_uint8)

print(f"✓ Masks saved to {MASKS_DIR}\n")

# ============================================================================
print("=" * 60)
print("STEP 4: Visualizing sample results (Frame 0)...")
print("=" * 60)

VIS_FRAME = 0
img = np.array(Image.open(os.path.join(FRAMES_DIR, frame_names[VIS_FRAME])))

plt.figure(figsize=(14, 10))
plt.imshow(img)

if VIS_FRAME in video_segments:
    for obj_id, mask in video_segments[VIS_FRAME].items():
        mask = np.asarray(mask)
        if mask.ndim == 3 and mask.shape[0] == 1:
            mask = mask[0]
        plt.imshow(mask, alpha=0.4, cmap='Greys')

plt.title(f"Frame {VIS_FRAME} - {len(video_segments.get(VIS_FRAME, {}))} tools visible")
plt.axis("off")
plt.show()
print(f"✓ Visualization complete\n")

# ============================================================================
print("=" * 60)
print("STEP 5: Exporting segmented video with color-coded overlays...")
print("=" * 60)

output_video = "segmented_video_multi_tools.mp4"
h, w = img.shape[:2]

tool_colors = [
    [255, 0, 0],      # Red
    [0, 255, 0],      # Green
    [0, 0, 255],      # Blue
    [255, 255, 0],    # Yellow
    [255, 0, 255],    # Magenta
    [0, 255, 255],    # Cyan
]

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*'mp4v'),
    30,
    (w, h)
)

for idx in range(len(frame_names)):
    frame = cv2.imread(os.path.join(FRAMES_DIR, frame_names[idx]))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    if idx in video_segments:
        overlay = frame.copy()
        for obj_id, mask in video_segments[idx].items():
            mask = np.asarray(mask)
            if mask.ndim == 3 and mask.shape[0] == 1:
                mask = mask[0]
            color = tool_colors[(obj_id - 1) % len(tool_colors)]
            overlay[mask] = color
        
        frame = cv2.addWeighted(overlay, 0.5, frame, 0.5, 0)
    
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

writer.release()
print(f"✓ Video exported: {output_video}")
print(f"✓ Total tools segmented: {len(clicked_points)}")

print("\n" + "=" * 60)
print("ALL STEPS COMPLETED SUCCESSFULLY!")
print("=" * 60)

## 13. Instrument Tip Keypoint Detection — Method

**Goal:** estimate the (x, y) coordinate of the **tip** of each instrument in every frame, in addition to the segmentation mask already produced by SAM2.

**Implementation note:** these cells read the results already saved to disk by Section 12 (`output/frames/frame_XXXXX.jpg` and `output/masks/frame_XXXXX.png`, where each mask pixel is `0` = background or the `obj_id` of the instrument). This means **you don't need `video_segments` in memory or to re-run the SAM2 propagation** (which takes hours) — you just need to have run Section 12 once, at some point, and kept the `output/` folder.

**Why this method:** we don't have a manually annotated keypoint dataset to train a supervised detector (e.g. a heatmap-regression network like HRNet/DeepLabCut). Instead, we derive the tip directly from the binary masks SAM2 already produces, with no extra training or data:

1. **Principal axis via PCA**: for each instrument mask, PCA is run on the (x, y) pixel coordinates. Surgical instruments are elongated (shaft-shaped), so the first principal component (highest variance) aligns with the instrument's longitudinal axis.
2. **Candidate endpoints**: all mask pixels are projected onto that axis, and the two extreme points (minimum and maximum projection) are taken. These are the instrument's two "ends": the tip and the end where it enters the surgical field (shaft/base, typically near the frame border, where it enters through the trocar).
3. **Tip vs. base disambiguation**:
   - In the first frame where each instrument appears, the endpoint closest to the image border is assigned as the **base**, and the other endpoint as the **tip** — assuming the instrument enters from outside the frame.
   - In subsequent frames, to prevent tip and base from swapping due to mask noise, the tip is chosen as whichever candidate endpoint is closest to the tip detected in the previous frame (temporal nearest-neighbor tracking).
4. **Limitations**: assumes rigid/elongated instruments with a single dominant tip per mask, and that the base is closer to the image border than the tip. With very curved or articulated instruments, a fully occluded tip, or two instruments crossing, this heuristic can fail. If a set of real tip annotations is created later, this method can serve as a pseudo-label generator to train a supervised keypoint detector (e.g. heatmap regression).


In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from collections import defaultdict

OUTPUT_FRAMES_DIR = os.path.join("output", "frames")
OUTPUT_MASKS_DIR = os.path.join("output", "masks")

mask_paths = sorted(glob.glob(os.path.join(OUTPUT_MASKS_DIR, "frame_*.png")))
if len(mask_paths) == 0:
    raise RuntimeError(
        f"No masks found in '{OUTPUT_MASKS_DIR}'. "
        "Run Section 12 (propagation) at least once to generate them."
    )
print(f"Masks found: {len(mask_paths)}")


def principal_axis_endpoints(mask):
    """Return the two extreme points along the principal axis (PCA) of a binary mask."""
    ys, xs = np.nonzero(mask)
    if len(xs) < 2:
        return None
    pts = np.column_stack([xs, ys]).astype(np.float32)
    mean = pts.mean(axis=0)
    centered = pts - mean
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    principal_axis = eigvecs[:, np.argmax(eigvals)]
    projections = centered @ principal_axis
    p_min = tuple(pts[np.argmin(projections)])
    p_max = tuple(pts[np.argmax(projections)])
    return p_min, p_max


def dist_to_border(point, w, h):
    """Minimum distance of a point to the image border (proxy for 'where the instrument enters')."""
    x, y = point
    return min(x, y, w - x, h - y)


tip_keypoints = defaultdict(dict)   # tip_keypoints[frame_idx][obj_id] = (x, y)
base_keypoints = defaultdict(dict)  # base_keypoints[frame_idx][obj_id] = (x, y)
prev_tip = {}                        # last known tip per obj_id, for temporal tracking

for mask_path in mask_paths:
    frame_idx = int(os.path.basename(mask_path).split('_')[1].split('.')[0])
    instance_mask = np.array(Image.open(mask_path))
    h_img, w_img = instance_mask.shape[:2]

    for obj_id in np.unique(instance_mask):
        obj_id = int(obj_id)
        if obj_id == 0:
            continue  # 0 = background

        mask = instance_mask == obj_id
        endpoints = principal_axis_endpoints(mask)
        if endpoints is None:
            continue
        p_a, p_b = endpoints

        if obj_id not in prev_tip:
            # first visible frame: the base is the endpoint closest to the image border
            if dist_to_border(p_a, w_img, h_img) < dist_to_border(p_b, w_img, h_img):
                base, tip = p_a, p_b
            else:
                base, tip = p_b, p_a
        else:
            # subsequent frames: the tip is the endpoint closest to the previous frame's tip
            d_a = np.hypot(p_a[0] - prev_tip[obj_id][0], p_a[1] - prev_tip[obj_id][1])
            d_b = np.hypot(p_b[0] - prev_tip[obj_id][0], p_b[1] - prev_tip[obj_id][1])
            tip, base = (p_a, p_b) if d_a < d_b else (p_b, p_a)

        tip_keypoints[frame_idx][obj_id] = tip
        base_keypoints[frame_idx][obj_id] = base
        prev_tip[obj_id] = tip

print(f"Tip keypoints computed for {len(tip_keypoints)} frames")

## 14. Visualize Keypoints on a Sample Frame

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

VIS_FRAME_KP = 0  # change this index to view a different frame

frame_path = os.path.join(OUTPUT_FRAMES_DIR, f"frame_{VIS_FRAME_KP:05d}.jpg")
mask_path = os.path.join(OUTPUT_MASKS_DIR, f"frame_{VIS_FRAME_KP:05d}.png")

img_kp = np.array(Image.open(frame_path))
instance_mask = np.array(Image.open(mask_path))

plt.figure(figsize=(14, 10))
plt.imshow(img_kp)

for obj_id in np.unique(instance_mask):
    obj_id = int(obj_id)
    if obj_id == 0:
        continue
    plt.imshow(instance_mask == obj_id, alpha=0.3, cmap='Greys')

    tip = tip_keypoints.get(VIS_FRAME_KP, {}).get(obj_id)
    base = base_keypoints.get(VIS_FRAME_KP, {}).get(obj_id)
    if tip is not None and base is not None:
        plt.plot([base[0], tip[0]], [base[1], tip[1]], '-', color='cyan', linewidth=2)
        plt.scatter(*tip, c='red', s=200, marker='*', edgecolors='white', linewidth=1.5, zorder=5,
                    label=f"Tool {obj_id} - tip")
        plt.scatter(*base, c='blue', s=120, marker='^', edgecolors='white', linewidth=1.5, zorder=5,
                    label=f"Tool {obj_id} - base")

plt.title(f"Frame {VIS_FRAME_KP} - Tip (red star) and base (blue triangle) per instrument")
plt.legend()
plt.axis("off")
plt.show()

## 15. Export Annotated Video with Keypoints

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

output_kp_video = "tip_pose_video.mp4"

tool_colors = [
    [255, 0, 0],      # Red
    [0, 255, 0],      # Green
    [0, 0, 255],      # Blue
    [255, 255, 0],    # Yellow
    [255, 0, 255],    # Magenta
    [0, 255, 255],    # Cyan
]

first_mask = np.array(Image.open(mask_paths[0]))
h_img, w_img = first_mask.shape[:2]

writer_kp = cv2.VideoWriter(
    output_kp_video,
    cv2.VideoWriter_fourcc(*'mp4v'),
    30,
    (w_img, h_img)
)

for mask_path in mask_paths:
    frame_idx = int(os.path.basename(mask_path).split('_')[1].split('.')[0])
    frame_path = os.path.join(OUTPUT_FRAMES_DIR, f"frame_{frame_idx:05d}.jpg")

    frame = cv2.imread(frame_path)
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    instance_mask = np.array(Image.open(mask_path))

    overlay = frame.copy()
    for obj_id in np.unique(instance_mask):
        obj_id = int(obj_id)
        if obj_id == 0:
            continue
        color = tool_colors[(obj_id - 1) % len(tool_colors)]
        overlay[instance_mask == obj_id] = color
    frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

    for obj_id, tip in tip_keypoints.get(frame_idx, {}).items():
        base = base_keypoints[frame_idx][obj_id]
        tip_xy = (int(tip[0]), int(tip[1]))
        base_xy = (int(base[0]), int(base[1]))
        cv2.arrowedLine(frame, base_xy, tip_xy, (255, 255, 0), 2, tipLength=0.15)  # yellow (RGB)
        cv2.circle(frame, tip_xy, 8, (255, 0, 0), -1)  # red (RGB)
        cv2.putText(frame, f"tip {obj_id}", (tip_xy[0] + 10, tip_xy[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    writer_kp.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

writer_kp.release()
print(f"Video with keypoints exported: {output_kp_video}")

## 16. Save Keypoints to CSV

In [ ]:
import csv

keypoints_csv_path = "tip_pose_keypoints.csv"

with open(keypoints_csv_path, "w", newline="") as f:
    writer_csv = csv.writer(f)
    writer_csv.writerow(["frame_idx", "obj_id", "tip_x", "tip_y", "base_x", "base_y"])
    for frame_idx in sorted(tip_keypoints.keys()):
        for obj_id, tip in tip_keypoints[frame_idx].items():
            base = base_keypoints[frame_idx][obj_id]
            writer_csv.writerow([frame_idx, obj_id, tip[0], tip[1], base[0], base[1]])

print(f"Keypoints saved to: {keypoints_csv_path}")
print(f"Total rows: {sum(len(v) for v in tip_keypoints.values())}")